Makes Figure 1 in the Faint Hot Components paper, as well as a figure for a talk (one region example over AIA). 

Presently, NuSTAR data is shifted to coalign with AIA. For JSOC AIA regions, the imput shift is saves in the targets directory. FOr NCCS AIA regions, the shift needed to be explicitly extracted from the difference between the AIA and NuSTAR region files. Code to do that is after the figure making cell (intentional, so it will fail and then you will go look at what's being done there to keep it in mind). 

In [ ]:
import matplotlib.pyplot as plt
from astropy.coordinates import SkyCoord, SkyOffsetFrame
from regions import CircleSkyRegion
import numpy as np
import glob

#Path to top-level do-dem directory - edit for your system.
path_to_dodem = '/Users/jmdunca2/do-dem/'
from sys import path as sys_path
sys_path.append(path_to_dodem+'/dodem/')

# #import nustar_dem_prep as nu
import images_and_coalignment as iac
import time_interval_selection as tis
import nustar_utilities as nuutil
import gauss2D as g2d
import nustar_dem_prep as nu


# import os
# # For macOS
# def play_sound():
#     os.system("afplay /System/Library/Sounds/Glass.aiff")

import pickle
import pathlib
import importlib
from astropy import units as u
import numpy as np


with open('/Users/jmdunca2/do-dem/reference_files/all_targets_postghost_postshut.pickle', 'rb') as f:
    data = pickle.load(f)

keys = list(data.keys())
print(keys)
print(len(keys))

with open('/Users/jmdunca2/do-dem/reference_files/samesames.pickle', 'rb') as f:
    data2 = pickle.load(f)

samesames = data2['same region lists']
#print(samesames)
ids=[]
for ss in samesames:
    id=ss[0].split(' ')[0]
    #print(id)
    if not id in ids:
        ids.append(id)

print(ids)
print(len(ids))
    

In [ ]:
importlib.reload(iac)
aia_dir = '/Users/jmdunca2/sample_aia/sample_aia/'

all_maps=[]
for k in ids:
    print(k)
    if k in ['01-nov-14_2', '22-apr-16_1', '22-nov-21_2', '03-jun-22_1', '18-mar-23_2']:
        continue

    ARDict = data[k]

    id_dirs = ARDict['datapaths']
    aiamaps = iac.get_orbit_aiamaps(aia_dir, [id_dirs[0]], wave=94)   
    all_maps.append(aiamaps[0])
    
    # obsids = ARDict['obsids']
    # working_dir = ARDict['working_dir']

    # if method=='double':
    #     gauss_stats = ARDict['gauss_stats']
    #     sep_axis = gauss_stats[0][0]
    # else:
    #     sep_axis = ''

In [ ]:
all_regiondicts = []

for k in ids:
    
    regiondicts=[]
    
    if k in ['01-nov-14_2', '22-apr-16_1', '22-nov-21_2', '03-jun-22_1',  '18-mar-23_2']:
        continue
    # if k in ['27-feb-22', '18-mar-23_2', '21-nov-21']:
    #     continue

    ARDict = data[k]
    method = ARDict['method']
    print(k)

    working_dir = ARDict['working_dir']
    ARDict['prepped_aia'] = working_dir+'all_aia_dicts_'+k+'_post/'
    
    orbdirs = glob.glob(ARDict['prepped_aia']+'*')
    orbdirs.sort()
    print(orbdirs)
    print('')
    
    if len(orbdirs)==0:
        for rd in ARDict['regdicts']:
            print(rd)
            print(rd[0])
            regiondicts.append(rd[0])
        print('')
        print('')
        #regiondict['centerx'],regiondict['centery'] #w/ arcsec units
        #regiondict['radius'] #w/out units
        #continue
    else:
        timefiles = glob.glob(orbdirs[0]+'/*')
        timefiles.sort()
        print(timefiles[0])
    
        with open(timefiles[0], 'rb') as f:
            aiadata = pickle.load(f)
    
        if 'region0' in aiadata.keys() and k != '29-may-18_2':
            try:
                print('Reg 0: ', aiadata['region0'].keys())
            except AttributeError:
                #for i in range(1,14):
                with open(timefiles[12], 'rb') as f:
                    aiadata = pickle.load(f)
                print('Reg 0: ', aiadata['region0'].keys())
    
            regiondicts.append(aiadata['region0'])
                #print('Reg 0: ', aiadata['region0'].keys())
                
    
        if 'region1' in aiadata.keys():
            print('Reg 1: ', aiadata['region1'].keys())
            regiondicts.append(aiadata['region1'])
    
        if 'radius' in aiadata.keys():
            print(aiadata.keys())
            regiondicts.append(aiadata)


    all_regiondicts.append(regiondicts)


    print('')

In [ ]:
#Get nustar maps

from astropy.io import fits
import nustar_pysolar as nustar

nu_smaps=[]
nu_keys=[]
for k in ids:
    
    regiondicts=[]
    
    if k in ['01-nov-14_2', '22-apr-16_1', '22-nov-21_2', '03-jun-22_1',  '18-mar-23_2']:
        continue
    # if k in ['27-feb-22', '18-mar-23_2', '21-nov-21']:
    #     continue

    ARDict = data[k]
    id_dirs = ARDict['datapaths']

    sunpos = glob.glob(id_dirs[0]+'/event_cl/*sunpos*.evt')
    count=1

    s = sunpos[0]

    with fits.open(s, ignore_missing_simple=True) as hdu:
        hdr = hdu[1].header
    time0 = nuutil.convert_nustar_time(hdr['TSTART'])
    time1 = nuutil.convert_nustar_time(hdr['TSTOP'])
    
    with fits.open(s) as hdu:
        evt_data = hdu[1].data
        hdr = hdu[1].header
    
    cleanevt = nustar.filter.event_filter(evt_data, energy_low=2.5, energy_high=10.,
                                     no_bad_pix_filter=True, no_grade_filter=True)
    
    nustar_map = nustar.map.make_sunpy(cleanevt, hdr)
    bl_ = SkyCoord( *(-1650, -1650)*u.arcsec, frame=nustar_map.coordinate_frame)
    tr_ = SkyCoord( *(1650, 1650)*u.arcsec, frame=nustar_map.coordinate_frame)
    nu_smap = nustar_map.submap(bottom_left=bl_, top_right=tr_)
    nu_smaps.append(nu_smap)
    nu_keys.append(k)


In [ ]:
import visualize_dem_results as viz
from sunpy.coordinates import Helioprojective
import importlib
import sunpy.map
import glob

from scipy import ndimage
from scipy.ndimage.interpolation import shift


#noindices = [0, 5, 9, 14, 15, 19, 20, 21]
noindices = [4]
#noindices = []

fig = plt.figure(figsize=(40,22))

bads=0
howmany=1
nonshift_keys=[]
for i in range(0, len(all_regiondicts)):

    if i not in noindices:
        #print(i)

        m = all_maps[i]
        nu_smap = nu_smaps[i]

        try:
            shiftt = data[nu_keys[i]]['nushift']
        except KeyError:
            nonshift_keys.append(nu_keys[i])
            try:
                shiftt = data[nu_keys[i]]['found_shift']
            except KeyError:
                shiftt = [0, 0]
                bads+=1
            
        #Note: axes have same scale, used axis1 for both coordinates
        xshift=shiftt[0]/nu_smap.scale.axis1.value
        yshift=shiftt[1]/nu_smap.scale.axis1.value

        #Making shifted NuSTAR submap
        shifted_data = shift(nu_smap.data, [yshift, xshift], mode='constant')
        shift_nu_smap=sunpy.map.Map(shifted_data, nu_smap.meta)

        nu_smap = shift_nu_smap
  
    
        regd = all_regiondicts[i]
        if len(regd) == 1:
            xx = regd[0]['centerx'].value
            yy = regd[0]['centery'].value
        else:
            xx = (regd[0]['centerx'].value + regd[1]['centerx'].value)/2.
            yy = (regd[0]['centery'].value + regd[1]['centery'].value)/2.
            
    
        #Set broad box for plotting (using region object)
        bl=[(xx-500)*u.arcsec, (yy-500)*u.arcsec]
        tr=[(xx+500)*u.arcsec,(yy+500)*u.arcsec]
        #print(tr[0]-bl[0], tr[1]-bl[1])
    
        bottom_left = SkyCoord(bl[0]-150*u.arcsec, bl[1]-150*u.arcsec, frame=m.coordinate_frame)
        top_right = SkyCoord(tr[0]+150*u.arcsec,tr[1]+150*u.arcsec, frame=m.coordinate_frame)
        mm = m.submap(bottom_left=bottom_left, top_right=top_right)

        #Make a new observer object, using the observer from the AIA map
        new_observer = mm.observer_coordinate

        #Note: for some reason, using the same data shape with a reprojection can cause most of the data to 
        #be cropped out and set to NaN values.
        #To be safe, we will use a VERY large output shape.
        #out_shape = nu_smap.data.shape
        out_shape = (1500,1500)
        
        out_ref_coord = SkyCoord(0*u.arcsec, 0*u.arcsec, obstime=new_observer.obstime,
                                 frame='helioprojective', observer=new_observer,
                                 rsun=nu_smap.coordinate_frame.rsun)

        out_header = sunpy.map.make_fitswcs_header(
            out_shape,
            out_ref_coord,
            scale=u.Quantity(nu_smap.scale)
            )


        with Helioprojective.assume_spherical_screen(nu_smap.observer_coordinate):
            nu_reproject = nu_smap.reproject_to(out_header)

        data2 = ndimage.gaussian_filter(nu_reproject.data, 8)
        nu_reproject2 = sunpy.map.Map(data2, nu_reproject.meta)


        #Making plot limits
        world_coords = SkyCoord(Tx=[bl[0],tr[0]], Ty=[bl[1],tr[1]], frame=mm.coordinate_frame)
        apixel_coords_x, apixel_coords_y = mm.wcs.world_to_pixel(world_coords)


        contourlevs = [1, 5, 10, 20,50,70,90]
        colors = ['orange', 'blue', 'teal', 'indianred', 'green', 'pink', 'purple', 'gold', 'yellow', 'green']
        
        #ax = fig.add_subplot(6,8,howmany, projection=mm)
        ax = fig.add_subplot(4,7,howmany, projection=mm)
        #ax.set_axis_off()

        norm = mm.plot_settings['norm']
        #print(np.percentile(mm.data, [1, 99.9]))
        #print(norm.vmin, norm.vmax)
        #AIA94 norm
        norm.vmin, norm.vmax = 0.75, 45 
        #AIA131 norm
        #norm.vmin, norm.vmax = 0.75, 200
        #AIA211 norm
        #norm.vmin, norm.vmax = 0.75, 2500
        #AIA171 norm
        #norm.vmin, norm.vmax = 0.75, 3000
        
        mm.plot(axes=ax, norm=norm)
        levels = np.array(contourlevs)*u.percent 
        #print(levels)
        #print(np.min(np.diff(levels)))
        nu_reproject2.draw_contours(levels, axes=ax, alpha=1, zorder=1, colors='orange') #colors)
        #shifted_nu_smap.plot(axes=ax)
        
        ax.set_xlim(apixel_coords_x)
        ax.set_ylim(apixel_coords_y)

        lon = ax.coords[0]
        lat = ax.coords[1]

        lon.set_ticks_visible(False)
        lon.set_ticklabel_visible(False)
        lat.set_ticks_visible(False)
        lat.set_ticklabel_visible(False)
        lon.set_axislabel('')
        lat.set_axislabel('')
        

        ax.set_title(mm.name[0:6]+r' $\mathrm{\AA}$ '+mm.name[18:-8], fontsize=20)
        howmany+=1

        for r in regd:
            region = iac.make_region(r, mm)
            og_region = region.to_pixel(mm.wcs)                    
            og_region.plot(axes=ax, color='red', ls='--', lw=3)

        #print('')

if bads > 0:
    print("NEED TO GO RUN THE CODE BELOW TO GET NUSTAR SHIFTS FROM THE REGION FILES!")
        

plt.savefig('All_regions_AIA_reference_new94_edit.png', dpi=300)

In [ ]:
#Scale over which we are doing the gaussian smoothing
nu_reproject.scale[0].to(u.arcsec/u.pix)*8

In [ ]:
#Adding the found_shift entry to the targets dictionary by examining the shift between the AIA and NuSTAR region files. 
import glob
import region_fitting as rf
import nustar_utilities as nuutil

for nsk in nonshift_keys:
    print(nsk, data[nsk]['method'])
    #Get prepped AIA dictionary (containing AIA regions)
    ff = glob.glob(data[nsk]['prepped_aia']+'/*')
    ff.sort()
    fff = glob.glob(ff[0]+'/*')
    fff.sort()
    #print(fff[0])
    with open(fff[0], 'rb') as f:
         paia = pickle.load(f)
    #Get AIA region center X,Y
    if 'region0' in paia.keys():
        xx, yy = paia['region0']['centerx'], paia['region0']['centery']
    else:
        xx, yy = paia['centerx'], paia['centery']

    all_all_time_intervals = data[nsk]['per_region_all_time_intervals']
    orbit_ind=0
    
    if data[nsk]['method']=='fit':
        all_time_intervals = all_all_time_intervals[0]
        time_interval = all_time_intervals[orbit_ind][0]
        time0, time1 = time_interval

        time = time_interval
        timestring = time[0].strftime('%H-%M-%S')
        stopstring = time[1].strftime('%H-%M-%S')
        timestring=timestring+'_'+stopstring
        print(timestring)

        rfs = glob.glob(data[nsk]['working_dir']+timestring+'/'+'*A*.reg')[0]
            
        print(rfs)
        print(time0, time1)
        offset, rad = rf.read_regfile(rfs, time0, time1, 'hourangle')
        #print('NuSTAR Region: ', offset.value)
            
        #print('AIA Region: ', xx.value,yy.value)
        #print('AIA-NuSTAR = ', xx.value-offset[0].value, yy.value-offset[1].value)
        foundshift = [xx.value-offset[0].value, yy.value-offset[1].value]
        #print('SHifted NuSTAR: ', offset[0].value + foundshift[0], offset[1].value + foundshift[1])
        print('Found Shift: ', foundshift)

    if data[nsk]['method']=='double':

        first_intervals = [at[orbit_ind][0] for at in all_all_time_intervals]
        durations = [(fi[1]-fi[0]).to(u.s).value for fi in first_intervals]
        maxint = np.argmax(durations)
        time_interval = first_intervals[maxint]
        time0, time1 = time_interval

        
        regpath = ('/').join(data[nsk]['res_file_dict(s)'][0]['quiet files all-inst'][0].split('/')[0:-4])+'/'
        #print(regpath)
        rfs_ = glob.glob(regpath+'*A_0.reg')
        rfs_.sort()
        rfs = rfs_[0]
        
        offset, rad = rf.read_regfile(rfs, time0, time1, 'hourangle')
        print('NuSTAR Region: ', offset.value)
            
        print('AIA Region: ', xx.value,yy.value)
        #print('AIA-NuSTAR = ', xx.value-offset[0].value, yy.value-offset[1].value)
        foundshift = [xx.value-offset[0].value, yy.value-offset[1].value]
        #print('SHifted NuSTAR: ', offset[0].value + foundshift[0], offset[1].value + foundshift[1])
        print('Found Shift: ', foundshift)    

    data[nsk]['found_shift'] = foundshift
    print('')

In [ ]:
#Making a single region figure for Hinode-19 talk cover page. 

import visualize_dem_results as viz
from sunpy.coordinates import Helioprojective
import importlib
import sunpy.map
import glob

from scipy import ndimage
from scipy.ndimage.interpolation import shift


#noindices = [0, 5, 9, 14, 15, 19, 20, 21]
noindices = [4]
#noindices = []

fig = plt.figure(figsize=(10,10))

howmany=1
nonshift_keys=[]
i=21
#print(i)

m = all_maps[i]
nu_smap = nu_smaps[i]

try:
    shiftt = data[nu_keys[i]]['nushift']
except KeyError:
    nonshift_keys.append(nu_keys[i])
    try:
        shiftt = data[nu_keys[i]]['found_shift']
    except KeyError:
        shiftt=[0,0]
    
#Note: axes have same scale, used axis1 for both coordinates
xshift=shiftt[0]/nu_smap.scale.axis1.value
yshift=shiftt[1]/nu_smap.scale.axis1.value

#Making shifted NuSTAR submap
shifted_data = shift(nu_smap.data, [yshift, xshift], mode='constant')
shift_nu_smap=sunpy.map.Map(shifted_data, nu_smap.meta)

nu_smap = shift_nu_smap


regd = all_regiondicts[i]
if len(regd) == 1:
    xx = regd[0]['centerx'].value
    yy = regd[0]['centery'].value
else:
    xx = (regd[0]['centerx'].value + regd[1]['centerx'].value)/2.
    yy = (regd[0]['centery'].value + regd[1]['centery'].value)/2.
    

#Set broad box for plotting (using region object)
bl=[(xx-500)*u.arcsec, (yy-500)*u.arcsec]
tr=[(xx+500)*u.arcsec,(yy+1000)*u.arcsec]
#print(tr[0]-bl[0], tr[1]-bl[1])

bottom_left = SkyCoord(bl[0]-150*u.arcsec, bl[1]-150*u.arcsec, frame=m.coordinate_frame)
top_right = SkyCoord(tr[0]+150*u.arcsec,tr[1]+150*u.arcsec, frame=m.coordinate_frame)
mm = m.submap(bottom_left=bottom_left, top_right=top_right)

#Make a new observer object, using the observer from the AIA map
new_observer = mm.observer_coordinate

#Note: for some reason, using the same data shape with a reprojection can cause most of the data to 
#be cropped out and set to NaN values.
#To be safe, we will use a VERY large output shape.
#out_shape = nu_smap.data.shape
out_shape = (1500,1500)

out_ref_coord = SkyCoord(0*u.arcsec, 0*u.arcsec, obstime=new_observer.obstime,
                         frame='helioprojective', observer=new_observer,
                         rsun=nu_smap.coordinate_frame.rsun)

out_header = sunpy.map.make_fitswcs_header(
    out_shape,
    out_ref_coord,
    scale=u.Quantity(nu_smap.scale)
    )


with Helioprojective.assume_spherical_screen(nu_smap.observer_coordinate):
    nu_reproject = nu_smap.reproject_to(out_header)

data2 = ndimage.gaussian_filter(nu_reproject.data, 8)
nu_reproject2 = sunpy.map.Map(data2, nu_reproject.meta)


#Making plot limits
world_coords = SkyCoord(Tx=[bl[0],tr[0]], Ty=[bl[1],tr[1]], frame=mm.coordinate_frame)
apixel_coords_x, apixel_coords_y = mm.wcs.world_to_pixel(world_coords)


contourlevs = [1, 5, 10, 20,50,70,90]
colors = ['orange', 'blue', 'teal', 'indianred', 'green', 'pink', 'purple', 'gold', 'yellow', 'green']

#ax = fig.add_subplot(6,8,howmany, projection=mm)
ax = fig.add_subplot(1,1,howmany, projection=mm)
#ax.set_axis_off()

norm = mm.plot_settings['norm']
#print(np.percentile(mm.data, [1, 99.9]))
#print(norm.vmin, norm.vmax)
#AIA94 norm
#norm.vmin, norm.vmax = 0.75, 45 
#AIA131 norm
norm.vmin, norm.vmax = 0.75, 200
#AIA211 norm
#norm.vmin, norm.vmax = 0.75, 2500
#AIA171 norm
#norm.vmin, norm.vmax = 0.75, 3000

mm.plot(axes=ax, norm=norm)
levels = np.array(contourlevs)*u.percent 
#print(levels)
#print(np.min(np.diff(levels)))
nu_reproject2.draw_contours(levels, axes=ax, alpha=1, zorder=1, colors='orange') #colors)
#shifted_nu_smap.plot(axes=ax)

ax.set_xlim(apixel_coords_x)
ax.set_ylim(apixel_coords_y)

lon = ax.coords[0]
lat = ax.coords[1]

lon.set_ticks_visible(False)
lon.set_ticklabel_visible(False)
lat.set_ticks_visible(False)
lat.set_ticklabel_visible(False)
lon.set_axislabel('')
lat.set_axislabel('')

ax.set_axis_off()
ax.set_title("")


#ax.set_title(mm.name[0:6]+r' $\mathrm{\AA}$ '+mm.name[18:-8], fontsize=20)
howmany+=1

# for r in regd:
#     region = iac.make_region(r, mm)
#     og_region = region.to_pixel(mm.wcs)                    
#     og_region.plot(axes=ax, color='red', ls='--', lw=3)

#print('')
        
    

plt.savefig('20july2021_singleplot.png', dpi=300)

In [ ]:
from importlib.metadata import version
print(version('sunpy'))

In [ ]:
# sks = ['03-may-21_1 region_0', '08-jan-21 region_0', '26-jul-16_1 region_1', '09-sep-18 region_0', '02-sep-15 region_0', 
#        '10-sep-18 region_0', '01-sep-15 region_0', '29-apr-21 region_0', '02-sep-15 region_1', '29-jan-20 region_0', 
#        '29-may-18_2 region_0', '11-sep-17 region_0', '12-sep-17 region_1', '12-sep-17 region_0', '13-sep-17 region_0', 
#        '13-sep-17 region_1', '03-may-21_2 region_0', '06-jun-20 region_0', '07-jun-20 region_0', '08-jun-20 region_0', 
#        '26-jul-16_1 region_0', '27-jul-16_1 region_0', '29-may-18_1 region_0', '26-jul-16_2 region_0', '12-apr-19 region_0', 
#        '22-apr-16_2 region_0', '13-apr-19 region_0', '19-feb-16 region_0', '10-oct-17 region_1', '20-jul-21 region_0', 
#        '20-jul-21 region_1', '10-oct-17 region_0', '30-jul-21_1 region_1', '30-jul-21_1 region_0', '30-jul-21_2 region_0', 
#        '20-jan-21 region_0']

# kks = [s.split(' ')[0] for s in sks]

# if '22-apr-16_1' in keys:
#     keys.remove('22-apr-16_1')

# noindices = [0, 5, 9, 14, 15, 19, 20, 21]
# yesindices=[]
# for ind in range(0, len(keys)-1):
#     if ind not in noindices:
#         yesindices.append(ind)

# yeskeys=[keys[i] for i in yesindices]
# sard = [all_regiondicts[i] for i in yesindices]
# smaps = [all_maps[i] for i in yesindices]

# orderinds=[]
# for kk in kks:
#     if kk in yeskeys:
#         nind = yeskeys.index(kk)
#         if nind not in orderinds:
#             orderinds.append(nind)


# orderkeys = [yeskeys[i] for i in orderinds]
# osard = [sard[i] for i in orderinds]
# osmaps = [smaps[i] for i in orderinds]


# fig = plt.figure(figsize=(40,28))

# howmany=1
# for i in range(0, len(osard)):

#     m = osmaps[i]

#     regd = osard[i]
#     if len(regd) == 1:
#         xx = regd[0]['centerx'].value
#         yy = regd[0]['centery'].value
#     else:
#         xx = (regd[0]['centerx'].value + regd[1]['centerx'].value)/2.
#         yy = (regd[0]['centery'].value + regd[1]['centery'].value)/2.
        

#     #Set broad box for plotting (using region object)
#     bl=[(xx-275)*u.arcsec, (yy-275)*u.arcsec]
#     tr=[(xx+275)*u.arcsec,(yy+275)*u.arcsec]
#     #print(tr[0]-bl[0], tr[1]-bl[1])

#     bottom_left = SkyCoord(bl[0]-100*u.arcsec, bl[1]-100*u.arcsec, frame=m.coordinate_frame)
#     top_right = SkyCoord(tr[0]+100*u.arcsec,tr[1]+100*u.arcsec, frame=m.coordinate_frame)
#     mm = m.submap(bottom_left=bottom_left, top_right=top_right)

    
#     ax = fig.add_subplot(4,6,howmany, projection=mm)
#     ax.set_axis_off()

#     norm = mm.plot_settings['norm']
#     print(norm.vmin, norm.vmax)
#     #norm.vmin, norm.vmax = 0.75, 60 #np.percentile(mm.data, [1, 99.9])
#     #print(norm.vmin, norm.vmax)
    
#     mm.plot(axes=ax, norm=norm)
#     ax.set_title(mm.name[0:6]+r'$\AA$'+mm.name[17:-3], fontsize=20)
#     howmany+=1

#     for r in regd:
#         region = iac.make_region(r, mm)
#         og_region = region.to_pixel(mm.wcs)                    
#         og_region.plot(axes=ax, color='red', ls='--', lw=3)

#     #print('')
        
        

# plt.savefig('Age_sorted_all_regions_AIA211_reference.png')


In [ ]:
# #keys.remove('22-apr-16_1')

# noindices = [0, 5, 9, 14, 15, 19, 20, 21]
# yesindices=[]
# for ind in range(0, len(keys)-1):
#     if ind not in noindices:
#         yesindices.append(ind)

# yeskeys=[keys[i] for i in yesindices]

In [ ]:
# sard = [all_regiondicts[i] for i in yesindices]
# smaps = [all_maps[i] for i in yesindices]

# fig = plt.figure(figsize=(40,28))

# howmany=1
# for i in range(0, len(sard)):

#     m = smaps[i]

#     regd = sard[i]
#     if len(regd) == 1:
#         xx = regd[0]['centerx'].value
#         yy = regd[0]['centery'].value
#     else:
#         xx = (regd[0]['centerx'].value + regd[1]['centerx'].value)/2.
#         yy = (regd[0]['centery'].value + regd[1]['centery'].value)/2.
        

#     #Set broad box for plotting (using region object)
#     bl=[(xx-275)*u.arcsec, (yy-275)*u.arcsec]
#     tr=[(xx+275)*u.arcsec,(yy+275)*u.arcsec]
#     #print(tr[0]-bl[0], tr[1]-bl[1])

#     bottom_left = SkyCoord(bl[0]-100*u.arcsec, bl[1]-100*u.arcsec, frame=m.coordinate_frame)
#     top_right = SkyCoord(tr[0]+100*u.arcsec,tr[1]+100*u.arcsec, frame=m.coordinate_frame)
#     mm = m.submap(bottom_left=bottom_left, top_right=top_right)

    
#     ax = fig.add_subplot(4,6,howmany, projection=mm)
#     ax.set_axis_off()

#     norm = mm.plot_settings['norm']
#     #print(norm.vmin, norm.vmax)
#     norm.vmin, norm.vmax = 0.75, 60 #np.percentile(mm.data, [1, 99.9])
#     #print(norm.vmin, norm.vmax)
    
#     mm.plot(axes=ax, norm=norm)
#     ax.set_title(mm.name[0:6]+r'$\AA$'+mm.name[17:-3], fontsize=20)
#     howmany+=1

#     for r in regd:
#         region = iac.make_region(r, mm)
#         og_region = region.to_pixel(mm.wcs)                    
#         og_region.plot(axes=ax, color='red', ls='--', lw=3)

#     #print('')
        
        

# plt.savefig('Age_sorted_all_regions_AIA94_reference.png')